# Учебное исследование территории: Нижневартовск (ХМАО — Югра)

Ноутбук собран по мотивам `examples/pipeline.ipynb` из BlocksNet и адаптирован под:

- BlocksNet `1.0.0a9`;
- территорию **Нижневартовск, Ханты-Мансийский автономный округ — Югра, Россия**;
- два готовых графа: **автомобильный (`drive`)** и **пешеходный (`walk`)**;
- сервисы: **школы**, **детские сады**, **поликлиники**.

Порядок этапов сохранён как в pipeline:

1. загрузка данных и нарезка кварталов;
2. назначение функционального использования;
3. построение графов и матрицы доступности;
4. агрегация зданий;
5. агрегация сервисов;
6. расчёт показателей: обеспеченность, плотность, морфотипы, зональная доступность;
7. сохранение результатов и проверки.

Ноутбук рассчитан на последовательное выполнение **Run All**. Критические проверки останавливают расчёт с понятным сообщением; некритические печатают предупреждение и продолжают работу.


## Как устроен ноутбук

- Все параметры собраны в одной ячейке ниже: территория, графы, сервисы, кэш, порог времени.
- `USE_CACHE = True` — при повторном запуске исходные OSM-слои, готовые графы и матрицы читаются из `examples/data/nizhnevartovsk/`. Кварталы, здания, сервисы и показатели пересчитываются, чтобы ноутбук оставался учебным.
- `REDOWNLOAD = True` — принудительно скачать OSM-слои, графы и пересчитать матрицы.
- Все векторные слои сохраняются в GeoJSON, матрицы — в Parquet, графы — в Pickle, сводные таблицы — в CSV.
- Итоговые GeoJSON и таблицы затем использует отдельный этап — HTML-дашборд.
- Первый запуск скачивает данные из OpenStreetMap и может занять несколько минут. Это нормально.
- Перед расчётами в ноутбуке есть пояснения: что означает показатель, в каких единицах измеряется и что помогает узнать о городе.

### Быстрый запуск в Windows PowerShell

```powershell
.\.venv\Scripts\Activate.ps1
python -m pip install -e ".[dev]"
python -m jupyter lab examples\blocksnet_nizhnevartovsk.ipynb
```

Если PowerShell не разрешает активацию окружения:

```powershell
Set-ExecutionPolicy -Scope Process -ExecutionPolicy Bypass
.\.venv\Scripts\Activate.ps1
```

В `pyproject.toml` BlocksNet установлен с extras `[ipynb,torch]`. `torch` нужен BlocksNet `1.0.0a9` даже для нарезки кварталов, поэтому после изменения зависимостей обязательно повторите `python -m pip install -e ".[dev]"`. Первая установка может занять несколько минут: PyTorch скачивается отдельно.


In [ ]:
import json
import pickle
import warnings
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="osmnx")

try:
    import blocksnet
    import iduedu
    from blocksnet.analysis.indicators import calculate_density_indicators
    from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes
    from blocksnet.analysis.network.accessibility import area_accessibility
    from blocksnet.analysis.provision import competitive_provision
    from blocksnet.blocks.aggregation import aggregate_objects
    from blocksnet.blocks.assignment import assign_land_use
    from blocksnet.blocks.cutting import cut_urban_blocks, preprocess_urban_objects
    from blocksnet.config import service_types_config
    from blocksnet.enums import LandUse
    from blocksnet.preprocessing.imputing import impute_buildings, impute_services
    from blocksnet.relations import calculate_accessibility_matrix, get_accessibility_graph
except ImportError as exc:
    raise ImportError(
        "Не установлены зависимости для ноутбука. В PowerShell выполните:\n"
        "  .\\.venv\\Scripts\\Activate.ps1\n"
        "  python -m pip install -e \".[dev]\"\n"
        f"Исходная ошибка: {exc}"
    ) from exc

TERRITORY_QUERY = "Нижневартовск, Ханты-Мансийский автономный округ — Югра, Россия"
TERRITORY_FALLBACK_QUERY = "Нижневартовск, Россия"
TERRITORY_SLUG = "nizhnevartovsk"
GRAPH_TYPES = ("drive", "walk")
SERVICES = ("school", "kindergarten", "polyclinic")
USE_CACHE = True
REDOWNLOAD = False
MAX_TRAVEL_MIN = 180
N_CLUSTERS = 11
RANDOM_STATE = 42
DEFAULT_LIVING_DEMAND = 30
MIN_BOUNDARY_KM2 = 1.0
MAX_BOUNDARY_KM2 = 2000.0


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_DIR = find_project_root(Path.cwd())
EXAMPLES_DIR = PROJECT_DIR / "examples"
DATA_DIR = EXAMPLES_DIR / "data" / TERRITORY_SLUG
OUTPUT_DIR = EXAMPLES_DIR / "outputs" / TERRITORY_SLUG
RAW_DIR = DATA_DIR / "raw"
GRAPH_DIR = DATA_DIR / "graphs"
MATRIX_DIR = DATA_DIR / "matrices"
SERVICE_DIR = DATA_DIR / "services"
BLOCK_DIR = DATA_DIR / "blocks"
RESULT_DIR = DATA_DIR / "results"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

for folder in (
    DATA_DIR,
    RAW_DIR,
    GRAPH_DIR,
    MATRIX_DIR,
    SERVICE_DIR,
    BLOCK_DIR,
    RESULT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
):
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (9, 9)
plt.rcParams["font.family"] = "DejaVu Sans"
pd.set_option("display.max_columns", 80)


def require(condition, message):
    if not condition:
        raise ValueError(f"ПРОВЕРКА НЕ ПРОЙДЕНА: {message}")


def warn(message):
    print(f"ПРЕДУПРЕЖДЕНИЕ: {message}")


def json_value(value):
    if hasattr(value, "value"):
        return value.value
    if value is None:
        return None
    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value, ensure_ascii=False)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def read_geo(path, crs=None, block_level=False):
    path = Path(path)
    try:
        gdf = gpd.read_file(path)
    except Exception:
        gdf = gpd.GeoDataFrame(geometry=[], crs=4326)
    if gdf.empty and gdf.crs is None:
        gdf = gdf.set_crs(4326)
    if block_level:
        if "block_id" in gdf.columns:
            gdf = gdf.set_index("block_id")
        elif len(gdf.columns) > 0:
            first = gdf.columns[0]
            gdf = gdf.rename(columns={first: "block_id"}).set_index("block_id")
    else:
        gdf = gdf.reset_index(drop=True)
    if crs is not None:
        gdf = gdf.to_crs(crs)
    return gdf


def save_geo(gdf, path, block_level=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    out = gdf.copy()
    if block_level:
        out = out.reset_index()
        if out.columns[0] != "block_id":
            out = out.rename(columns={out.columns[0]: "block_id"})
    else:
        out = out.reset_index(drop=True)
    for column in out.columns:
        if column == out.geometry.name:
            continue
        if pd.api.types.is_numeric_dtype(out[column].dtype):
            out[column] = out[column].replace([np.inf, -np.inf], np.nan)
        elif out[column].dtype == object:
            out[column] = out[column].map(json_value)
    if out.crs is not None and out.crs.to_epsg() != 4326:
        out = out.to_crs(4326)
    if out.empty:
        path.write_text(
            json.dumps({"type": "FeatureCollection", "features": []}, ensure_ascii=False),
            encoding="utf-8",
        )
    else:
        out.to_file(path, driver="GeoJSON")
    print(f"Сохранено: {path.relative_to(PROJECT_DIR)} ({len(out)} объектов)")


def describe_gdf(name, gdf):
    types = gdf.geom_type.value_counts().to_dict() if len(gdf) > 0 else {}
    print(f"{name}: {len(gdf)} объектов; CRS={gdf.crs}; типы={types}")


def keep_geometries(gdf, allowed):
    gdf = gdf.copy()
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf = gdf[gdf.geom_type.isin(allowed)].copy()
    return gdf.reset_index(drop=True)


def repair_geometries(gdf, allowed):
    gdf = gdf.copy()
    if len(gdf) == 0:
        return gdf
    gdf.geometry = gdf.geometry.make_valid()
    return keep_geometries(gdf, allowed)


def plot_map(gdf, column=None, title="", path=None, cmap="viridis", legend=True, figsize=(9, 9), **kwargs):
    fig, ax = plt.subplots(figsize=figsize)
    gdf.plot(
        ax=ax,
        column=column,
        cmap=cmap if column is not None else None,
        legend=legend and column is not None,
        edgecolor="white",
        linewidth=0.2,
        **kwargs,
    )
    ax.set_title(title)
    ax.set_axis_off()
    if path is not None:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


def fetch_osm_layer(name, tags, boundaries_4326, columns=None):
    path = RAW_DIR / f"{name}.geojson"
    if USE_CACHE and not REDOWNLOAD and path.exists():
        gdf = read_geo(path)
        print(f"Из кэша: {path.relative_to(PROJECT_DIR)}")
    else:
        try:
            gdf = ox.features_from_polygon(boundaries_4326.union_all(), tags=tags)
        except Exception as exc:
            warn(f"Не удалось загрузить слой {name}: {exc}")
            gdf = gpd.GeoDataFrame(geometry=[], crs=4326)
        gdf = gdf.reset_index(drop=True)
        if gdf.crs is None:
            gdf = gdf.set_crs(4326)
        else:
            gdf = gdf.to_crs(4326)
        if len(gdf) > 0 and columns is not None:
            keep = [c for c in columns if c in gdf.columns]
            if gdf.geometry.name not in keep:
                keep.append(gdf.geometry.name)
            gdf = gdf[keep].copy()
        save_geo(gdf, path)
    return gdf


print("Версии:")
print("  blocksnet:", getattr(blocksnet, "__version__", "неизвестно"))
print("  iduedu:", getattr(iduedu, "__version__", "неизвестно"))
print("  osmnx:", ox.__version__)
print("  geopandas:", gpd.__version__)
print("  pandas:", pd.__version__)
print("  networkx:", nx.__version__)
print("Проект:", PROJECT_DIR)
print("Данные:", DATA_DIR)
print("Результаты:", OUTPUT_DIR)


## Справочник показателей

| Показатель | Единицы | Что помогает узнать |
|---|---|---|
| `population` | чел. | Оценка численности жителей квартала по жилой площади |
| `count_buildings` | шт. | Сколько зданий попало в квартал |
| `fsi` | м² пола / м² участка | Интенсивность застройки: сколько площади пола приходится на участок |
| `gsi` | м² застройки / м² участка | Доля участка, занятая зданиями |
| `mxi` | м² жилья / м² пола | Насколько квартал жилой, а не деловой/сервисный |
| `l` | этажи | Средняя этажность застройки |
| `osr` | м² открытых пространств / м² пола | Сколько свободной от застройки площади приходится на площадь пола |
| `share_living`, `share_non_living` | отношение площадей | Соотношение жилой и нежилой площади в расчёте на площадь застройки |
| `land_use` | категория | Преобладающее функциональное использование квартала |
| `morphotype_spacematrix`, `morphotype_strelka` | категория | Тип городской среды по плотности, этажности и доле жилья |
| Матрица доступности | минуты | Минимальное время в пути между каждой парой кварталов |
| `area_accessibility_<graph>` | минуты | Средневзвешенное время поездки из квартала до всех кварталов города |
| `provision_strong_<service>` | доля 0–1 | Доля спроса, удовлетворённая в пределах норматива доступности |
| `provision_weak_<service>` | доля 0–1 | Доля спроса с учётом объектов за пределами норматива |
| `demand_<service>` | чел. | Спрос на сервис по нормативу на 1000 жителей |
| `capacity_<service>` | места / посещения | Вместимость объектов сервиса в квартале |

Точные формулы и оговорки — в пояснениях перед соответствующими разделами.


## 1. Границы территории

Граница города запрашивается у OpenStreetMap через Nominatim (`osmnx.geocode_to_gdf`). Если основной запрос не сработал, используется запасной.

Проверки:

- граница не пустая;
- геометрия — полигон или мультиполигон;
- площадь в разумном диапазоне (1–2000 км²);
- для расчётов выбирается локальная метрическая проекция UTM, чтобы площади и расстояния были в метрах.

Если площадь сильно больше ожидаемой, геокодер мог вернуть весь муниципальный район. Тогда измените `TERRITORY_QUERY` на более точный запрос и удалите `raw/boundaries.geojson`.


In [ ]:
boundary_path = RAW_DIR / "boundaries.geojson"

if USE_CACHE and not REDOWNLOAD and boundary_path.exists():
    boundaries = read_geo(boundary_path)
    print(f"Границы из кэша: {boundary_path.relative_to(PROJECT_DIR)}")
else:
    try:
        boundaries = ox.geocode_to_gdf(TERRITORY_QUERY)
    except Exception as exc:
        warn(f"Не удалось геокодировать основной запрос: {exc}")
        boundaries = ox.geocode_to_gdf(TERRITORY_FALLBACK_QUERY)
    boundaries = boundaries.reset_index(drop=True)
    boundaries = keep_geometries(boundaries, ["Polygon", "MultiPolygon"])
    boundaries.geometry = boundaries.geometry.make_valid()
    boundaries = keep_geometries(boundaries, ["Polygon", "MultiPolygon"])
    save_geo(boundaries, boundary_path)

require(len(boundaries) > 0, "Границы территории не найдены")
boundaries = boundaries.to_crs(4326)
boundaries = keep_geometries(boundaries, ["Polygon", "MultiPolygon"])
require(len(boundaries) > 0, "После проверки геометрий границы территории пусты")

crs = boundaries.estimate_utm_crs()
boundaries = boundaries.to_crs(crs)
area_km2 = boundaries.area.sum() / 1e6
print(f"Локальная проекция: {crs.to_string()}")
print(f"Площадь территории: {area_km2:.2f} км²")

require(
    MIN_BOUNDARY_KM2 <= area_km2 <= MAX_BOUNDARY_KM2,
    f"Площадь территории {area_km2:.1f} км² вне ожидаемого диапазона "
    f"{MIN_BOUNDARY_KM2:.1f}–{MAX_BOUNDARY_KM2:.1f} км². Проверьте TERRITORY_QUERY.",
)
if area_km2 > 1500:
    warn("Территория очень большая; расчёт матриц может занять много времени и памяти.")

boundaries_4326 = boundaries.to_crs(4326)
describe_gdf("Границы", boundaries)
plot_map(
    boundaries,
    title=f"Граница территории: {area_km2:.1f} км²",
    path=FIGURE_DIR / "boundaries.png",
)


## 2. OSM-слои для нарезки кварталов

Для нарезки кварталов нужны линейные и площадные барьеры:

- вода (реки, озёра, водохранилища);
- дороги;
- железные дороги.

Теги заданы явно. Загрузка идёт через Overpass API внутри OSMnx. Если Overpass временно недоступен, ячейка напечатает предупреждение, а критическая проверка ниже остановит расчёт только при отсутствии дорог.

Все слои сохраняются в `raw/` и при повторном запуске читаются из кэша.


In [ ]:
bc_tags = {
    "water": {
        "natural": ["water", "bay", "strait"],
        "waterway": ["riverbank", "dock", "canal"],
        "landuse": ["reservoir", "basin"],
    },
    "roads": {
        "highway": [
            "motorway",
            "trunk",
            "primary",
            "secondary",
            "tertiary",
            "residential",
            "unclassified",
            "service",
            "living_street",
            "motorway_link",
            "trunk_link",
            "primary_link",
            "secondary_link",
            "tertiary_link",
        ]
    },
    "railways": {
        "railway": [
            "rail",
            "light_rail",
            "subway",
            "tram",
            "narrow_gauge",
            "preserved",
            "funicular",
        ]
    },
}

water = fetch_osm_layer("water", bc_tags["water"], boundaries_4326)
roads = fetch_osm_layer("roads", bc_tags["roads"], boundaries_4326)
railways = fetch_osm_layer("railways", bc_tags["railways"], boundaries_4326)

water = repair_geometries(water, ["Polygon", "MultiPolygon", "LineString", "MultiLineString"]).to_crs(crs)
roads = repair_geometries(roads, ["LineString", "MultiLineString"]).to_crs(crs)
railways = repair_geometries(railways, ["LineString", "MultiLineString"]).to_crs(crs)

require(len(roads) > 0, "Не загружены дороги; без них нельзя нарезать кварталы")
if len(water) == 0:
    warn("Вода не загружена; кварталы будут без водных барьеров.")
if len(railways) == 0:
    warn("Железные дороги не загружены; кварталы будут без железнодорожных барьеров.")

describe_gdf("Вода", water)
describe_gdf("Дороги", roads)
describe_gdf("Железные дороги", railways)


## 3. Нарезка кварталов

Кварталы создаются как замкнутые участки между дорогами, железными дорогами и водой.

Проверки:

- кварталы не пустые;
- все геометрии — полигоны;
- суммарная площадь кварталов покрывает заметную часть территории;
- у каждого квартала есть целочисленный `block_id`.

Слишком маленькие фрагменты (меньше 1 м²) удаляются как артефакты нарезки.


In [ ]:
lines, polygons = preprocess_urban_objects(roads, railways, water)
blocks = cut_urban_blocks(boundaries, lines, polygons)
blocks = blocks.reset_index(drop=True)
blocks = blocks.explode(index_parts=False).reset_index(drop=True)
blocks.geometry = blocks.geometry.make_valid()
blocks = blocks.explode(index_parts=False).reset_index(drop=True)
blocks = blocks[blocks.geometry.notna() & ~blocks.geometry.is_empty].copy()
blocks = blocks[blocks.geom_type == "Polygon"].copy()
blocks = blocks[blocks.area > 1.0].copy()

require(len(blocks) > 0, "Не удалось нарезать кварталы")
blocks["block_id"] = np.arange(len(blocks), dtype=int)
blocks = blocks.set_index("block_id")
blocks["site_area"] = blocks.area

coverage = blocks.area.sum() / boundaries.area.sum()
print(f"Кварталов: {len(blocks):,}")
print(f"Покрытие территории кварталами: {coverage:.1%}")
require(coverage > 0.1, f"Кварталы покрывают только {coverage:.1%} территории; проверьте границы и барьеры.")
if coverage < 0.5:
    warn("Кварталы покрывают менее половины территории; возможны пропуски из-за воды или ошибок нарезки.")
if coverage > 1.05:
    warn("Площадь кварталов больше площади территории; проверьте геометрии.")

save_geo(blocks, BLOCK_DIR / "blocks.geojson", block_level=True)
plot_map(
    blocks,
    title=f"Кварталы: {len(blocks):,}",
    path=FIGURE_DIR / "blocks.png",
    facecolor="#e8eef5",
)


## 4. Функциональные зоны и назначение land use

Функциональные зоны берутся из OSM-тега `landuse`. Они нужны, чтобы назначить каждому кварталу преобладающее использование: жилое, деловое, рекреационное, промышленное, транспортное, специальное или сельскохозяйственное.

### 4.1. Правила сопоставления

Правила ниже сопоставляют OSM `landuse` с перечислением `LandUse` BlocksNet. Для каждого квартала выбирается зона с наибольшей долей пересечения.

Проверка: доля кварталов с назначенным `land_use` должна быть не ниже 80 %. Если доля ниже, это не ошибка: в OSM может не хватать разметки. Но интерпретировать жилые/деловые кварталы нужно осторожно.

`land_use` — категория, а не число. Она помогает понять, где в городе жильё, деловые зоны, рекреация, промышленность и транспорт.


In [ ]:
functional_zones = fetch_osm_layer(
    "functional_zones",
    {"landuse": True},
    boundaries_4326,
    columns=["landuse"],
)
functional_zones = repair_geometries(functional_zones, ["Polygon", "MultiPolygon"]).to_crs(crs)

if "landuse" in functional_zones.columns:
    functional_zones = functional_zones[functional_zones["landuse"].notna()].copy()
    functional_zones["functional_zone"] = (
        functional_zones["landuse"].astype(str).str.strip().str.lower()
    )
else:
    functional_zones["functional_zone"] = None

functional_zones = functional_zones[["geometry", "functional_zone"]].copy()
functional_zones = functional_zones.reset_index(drop=True)
describe_gdf("Функциональные зоны", functional_zones)

if len(functional_zones) == 0:
    warn("Функциональные зоны не загружены; land_use кварталов останется неопределённым.")
else:
    plot_map(
        functional_zones,
        column="functional_zone",
        title="Функциональные зоны OSM",
        path=FIGURE_DIR / "functional_zones.png",
        legend=False,
    )


In [ ]:
rules = {
    "commercial": LandUse.BUSINESS,
    "retail": LandUse.BUSINESS,
    "industrial": LandUse.INDUSTRIAL,
    "cemetery": LandUse.SPECIAL,
    "garages": LandUse.INDUSTRIAL,
    "residential": LandUse.RESIDENTIAL,
    "grass": LandUse.RECREATION,
    "farmland": LandUse.AGRICULTURE,
    "construction": LandUse.SPECIAL,
    "brownfield": LandUse.INDUSTRIAL,
    "forest": LandUse.RECREATION,
    "recreation_ground": LandUse.RECREATION,
    "religious": LandUse.SPECIAL,
    "flowerbed": LandUse.RECREATION,
    "military": LandUse.SPECIAL,
    "landfill": LandUse.TRANSPORT,
    "railway": LandUse.TRANSPORT,
    "transport": LandUse.TRANSPORT,
    "education": LandUse.SPECIAL,
    "healthcare": LandUse.SPECIAL,
    "public": LandUse.SPECIAL,
    "mixed": LandUse.BUSINESS,
}

blocks_with_land_use = blocks.copy()
known_functional_zones = functional_zones[
    functional_zones["functional_zone"].isin(rules)
].reset_index(drop=True)

if len(known_functional_zones) > 0:
    blocks_with_land_use = assign_land_use(
        blocks_with_land_use,
        known_functional_zones,
        rules,
    )
else:
    if len(functional_zones) > 0:
        warn("Ни одна функциональная зона не сопоставлена с правилами LandUse.")
    blocks_with_land_use["land_use"] = None
    blocks_with_land_use["share"] = np.nan

if "land_use" not in blocks_with_land_use.columns:
    blocks_with_land_use["land_use"] = None

blocks_with_land_use["site_area"] = blocks_with_land_use.area
blocks_with_land_use["land_use_str"] = blocks_with_land_use["land_use"].map(json_value)
assigned_share = blocks_with_land_use["land_use"].notna().mean()
print(f"Доля кварталов с назначенным land_use: {assigned_share:.1%}")
if assigned_share < 0.8:
    warn("Менее 80% кварталов получили land_use; проверьте полноту functional_zones.")

if len(functional_zones) > 0:
    unknown_zones = sorted(set(functional_zones["functional_zone"].dropna()) - set(rules))
    if unknown_zones:
        print("Не распознаны правила для зон:", ", ".join(unknown_zones))

save_geo(
    blocks_with_land_use,
    BLOCK_DIR / "blocks_land_use.geojson",
    block_level=True,
)

if blocks_with_land_use["land_use_str"].notna().any():
    plot_map(
        blocks_with_land_use,
        column="land_use_str",
        title="Функциональное использование кварталов",
        path=FIGURE_DIR / "land_use.png",
        legend=False,
    )
else:
    warn("Нет назначенного land_use для карты.")


## 5. Графы доступности и матрица времени

Для времени в пути используются два готовых типа графов BlocksNet/IduEdu:

- **`drive`** — автомобильная сеть OpenStreetMap. Время ребра считается по длине и скорости: `maxspeed` из OSM или нормативная скорость класса дороги;
- **`walk`** — пешеходная сеть OpenStreetMap. Скорость 5 км/ч (83,3 м/мин).

**Источник данных:** OpenStreetMap, загрузка через Overpass API, обработка библиотекой IduEdu, версия которой закреплена в BlocksNet `1.0.0a9`.

**Что не используется:** интермодальный граф, общественный транспорт и GTFS. Это два независимых графа, каждый считается отдельно.

**Единица времени:** минуты. У каждого ребра есть вес `time_min`. Графы кэшируются в `graphs/`.

Проверки для каждого графа:

- это объект NetworkX;
- есть узлы и рёбра;
- есть CRS;
- у узлов есть координаты `x`, `y`;
- у рёбер есть `time_min`;
- все веса конечные и неотрицательные.


In [ ]:
graphs = {}

for graph_type in GRAPH_TYPES:
    graph_path = GRAPH_DIR / f"graph_{graph_type}.pickle"
    if USE_CACHE and not REDOWNLOAD and graph_path.exists():
        with open(graph_path, "rb") as file:
            graph = pickle.load(file)
        print(f"Граф {graph_type} из кэша: {graph_path.relative_to(PROJECT_DIR)}")
    else:
        graph = get_accessibility_graph(boundaries_4326, graph_type)
        with open(graph_path, "wb") as file:
            pickle.dump(graph, file)
        print(f"Граф {graph_type} сохранён: {graph_path.relative_to(PROJECT_DIR)}")

    require(isinstance(graph, nx.Graph), f"{graph_type}: объект не является графом NetworkX")
    require(graph.number_of_nodes() > 0, f"{graph_type}: в графе нет узлов")
    require(graph.number_of_edges() > 0, f"{graph_type}: в графе нет рёбер")
    require("crs" in graph.graph, f"{graph_type}: в графе нет CRS")
    require(
        all("x" in data and "y" in data for _, data in graph.nodes(data=True)),
        f"{graph_type}: у узлов нет координат x/y",
    )
    require(
        all("time_min" in data for _, _, data in graph.edges(data=True)),
        f"{graph_type}: у рёбер нет веса time_min",
    )

    edge_times = np.array(
        [data["time_min"] for _, _, data in graph.edges(data=True)],
        dtype=float,
    )
    require(
        np.all(np.isfinite(edge_times)) and np.all(edge_times >= 0),
        f"{graph_type}: веса time_min содержат NaN, inf или отрицательные значения",
    )

    graphs[graph_type] = graph
    print(
        f"{graph_type}: узлов={graph.number_of_nodes():,}, рёбер={graph.number_of_edges():,}, "
        f"время ребра: медиана={np.median(edge_times):.3f} мин, максимум={edge_times.max():.3f} мин"
    )


### 5.1. Матрица времени между кварталами

Матрица `A[i, j]` — минимальное время в минутах от квартала `i` до квартала `j` по выбранному графу. Значение `inf` означает, что путь не найден в пределах порога `MAX_TRAVEL_MIN`.

Для расчёта показателей недостижимые пары заменяются на `MAX_TRAVEL_MIN` (штрафное время). Это позволяет считать зональную доступность и обеспеченность без бесконечностей.

Проверки:

- размер матрицы совпадает с числом кварталов;
- индексы и столбцы совпадают с `block_id`;
- есть хотя бы часть достижимых пар;
- диагональ близка к нулю.

Матрицы сохраняются в Parquet: `matrices/accessibility_matrix_<graph>.parquet`.


In [ ]:
matrices = {}
matrices_capped = {}

for graph_type in GRAPH_TYPES:
    matrix_path = MATRIX_DIR / f"accessibility_matrix_{graph_type}.parquet"
    if USE_CACHE and not REDOWNLOAD and matrix_path.exists():
        matrix = pd.read_parquet(matrix_path)
        print(f"Матрица {graph_type} из кэша: {matrix_path.relative_to(PROJECT_DIR)}")
    else:
        matrix = calculate_accessibility_matrix(
            blocks,
            graphs[graph_type],
            weight_key="time_min",
            dtype=np.float32,
            threshold=int(MAX_TRAVEL_MIN),
        )
        matrix.to_parquet(matrix_path)
        print(f"Матрица {graph_type} сохранена: {matrix_path.relative_to(PROJECT_DIR)}")

    require(
        matrix.shape == (len(blocks), len(blocks)),
        f"{graph_type}: размер матрицы {matrix.shape} не совпадает с числом кварталов {len(blocks)}",
    )
    require(
        matrix.index.equals(blocks.index),
        f"{graph_type}: индекс матрицы не совпадает с block_id",
    )
    require(
        matrix.columns.equals(blocks.index),
        f"{graph_type}: столбцы матрицы не совпадают с block_id",
    )

    values = matrix.to_numpy()
    finite_share = float(np.isfinite(values).mean())
    require(finite_share > 0, f"{graph_type}: в матрице нет достижимых пар")
    if finite_share < 0.95:
        warn(
            f"{graph_type}: достижимо только {finite_share:.1%} пар. "
            "Возможна сильная фрагментация графа или слишком маленький MAX_TRAVEL_MIN."
        )

    diagonal = np.diag(values)
    if not np.allclose(diagonal, 0, atol=1e-3):
        warn(f"{graph_type}: диагональ матрицы не равна нулю; проверьте привязку кварталов к графу.")

    finite_values = values[np.isfinite(values)]
    print(
        f"{graph_type}: достижимо={finite_share:.1%}, "
        f"медиана={np.median(finite_values):.2f} мин, "
        f"95-й процентиль={np.percentile(finite_values, 95):.2f} мин, "
        f"максимум={finite_values.max():.2f} мин"
    )

    capped = matrix.replace([np.inf, -np.inf], float(MAX_TRAVEL_MIN)).fillna(float(MAX_TRAVEL_MIN))
    matrices[graph_type] = matrix
    matrices_capped[graph_type] = capped


## 6. Здания

Здания используются для оценки застройки, этажности, жилой и нежилой площади, населения и морфотипов.

Из OSM сохраняются:

- `building` — тип здания;
- `building:levels` — число этажей, если указано.

**Население — это оценка**, а не точная статистика: BlocksNet делит жилую площадь на норматив 30 м²/чел. Если в OSM не хватает этажности или типов зданий, оценка может отличаться от официальных данных.

Геометрии могут быть полигонами, точками и линиями. Площадь застройки для точек будет нулевой; такие объекты не дадут вклада в плотность, но не остановят расчёт.

Здания с тегом `building=yes` считаются потенциально жилыми: в OSM у таких объектов часто не указан точный тип, и без этого допущения население могло бы обнулиться.


In [ ]:
buildings = fetch_osm_layer(
    "buildings",
    {"building": True},
    boundaries_4326,
    columns=["building", "building:levels"],
)
buildings = repair_geometries(
    buildings,
    ["Polygon", "MultiPolygon", "Point", "LineString", "MultiLineString"],
).to_crs(crs)

require(len(buildings) > 0, "Не загружены здания")
describe_gdf("Здания", buildings)


### 6.1. Импутация и агрегация на кварталы

В OSM редко есть все нужные атрибуты. BlocksNet импутирует:

- площадь застройки — из геометрии;
- число этажей — по `building:levels`, иначе 1;
- площадь пола — этажность × площадь застройки;
- жилую площадь — 90 % площади пола для жилых зданий;
- нежилую площадь — остаток;
- население — жилая площадь / 30 м²/чел.

Затем здания агрегируются на кварталы. Проверка: суммарное население должно быть больше нуля.


In [ ]:
is_living_tags = {
    "yes",
    "apartments",
    "residential",
    "house",
    "detached",
    "terrace",
    "dormitory",
    "semi",
    "semidetached_house",
    "bungalow",
    "cabin",
    "farm",
    "houseboat",
    "static_caravan",
    "villa",
}

buildings = buildings.copy()
if "building" not in buildings.columns:
    buildings["building"] = "yes"
buildings["building"] = buildings["building"].fillna("yes").astype(str)


def is_living_building(value):
    return any(part.strip().lower() in is_living_tags for part in value.split(";"))


buildings["is_living"] = buildings["building"].map(is_living_building)

if "building:levels" in buildings.columns:
    levels = (
        buildings["building:levels"]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.extract(r"(\d+(?:\.\d+)?)")[0]
    )
    buildings["number_of_floors"] = pd.to_numeric(levels, errors="coerce")
else:
    buildings["number_of_floors"] = np.nan

buildings["footprint_area"] = 0.0
for column in ("build_floor_area", "living_area", "non_living_area", "population"):
    buildings[column] = np.nan

buildings = impute_buildings(
    buildings,
    default_number_of_floors=1,
    living_area_coefficient=0.9,
    default_living_demand=DEFAULT_LIVING_DEMAND,
)
buildings["is_living"] = buildings["is_living"].astype(bool)
buildings.loc[buildings["footprint_area"] <= 0, "population"] = 0

save_geo(buildings, RAW_DIR / "buildings_imputed.geojson")

buildings_agg = buildings[
    [
        "geometry",
        "number_of_floors",
        "footprint_area",
        "build_floor_area",
        "living_area",
        "non_living_area",
        "population",
    ]
].copy()

blocks_buildings, excess_buildings = aggregate_objects(blocks, buildings_agg)
blocks_buildings = blocks_buildings.rename(columns={"count": "count_buildings"})
blocks_buildings["site_area"] = blocks_buildings.area

population_total = blocks_buildings["population"].sum()
print(f"Зданий: {len(buildings):,}")
print(f"Зданий вне кварталов: {len(excess_buildings):,}")
print(f"Суммарное население (оценка): {population_total:,.0f} чел.")
require(population_total > 0, "Суммарное население равно нулю; проверьте здания и импутацию.")

save_geo(
    blocks_buildings,
    BLOCK_DIR / "blocks_buildings.geojson",
    block_level=True,
)

plot_map(
    blocks_buildings,
    column="population",
    title="Оценка населения по кварталам, чел.",
    path=FIGURE_DIR / "population.png",
    cmap="YlOrRd",
)


## 7. Сервисы

Загружаются три типа объектов:

- `amenity=school` — школы;
- `amenity=kindergarten` — детские сады;
- `amenity=polyclinic` — поликлиники.

**Вместимость** — это количество мест (для школ и садиков) или посещений (для поликлиник) в зависимости от сервиса. Если в OSM нет `capacity`, BlocksNet подставляет минимальный норматив для данного типа сервиса.

Если для какого-то типа объектов в OSM нет, ноутбук напечатает предупреждение и создаст нулевую обеспеченность. Это не остановит остальные расчёты.


In [ ]:
services_raw = {}

for service in SERVICES:
    gdf = fetch_osm_layer(
        f"service_{service}",
        {"amenity": service},
        boundaries_4326,
        columns=["amenity", "capacity", "name"],
    )
    gdf = repair_geometries(
        gdf,
        ["Point", "Polygon", "MultiPolygon", "LineString", "MultiLineString"],
    ).to_crs(crs)
    services_raw[service] = gdf
    if len(gdf) == 0:
        warn(f"В OSM не найдены объекты для сервиса {service}.")
    else:
        print(f"{service}: {len(gdf)} объектов")

describe_gdf("Школы", services_raw["school"])
describe_gdf("Детские сады", services_raw["kindergarten"])
describe_gdf("Поликлиники", services_raw["polyclinic"])


### 7.1. Вместимость и агрегация на кварталы

Вместимость суммируется по кварталам. Для каждого сервиса сохраняются:

- исходные/импутированные объекты: `services/<service>.geojson`;
- агрегаты по кварталам: `blocks/blocks_<service>.geojson`.

`count_<service>` — число объектов в квартале, `capacity_<service>` — суммарная вместимость.


In [ ]:
services = {}
blocks_services = {}

for service in SERVICES:
    gdf = services_raw[service].copy()
    if "capacity" not in gdf.columns:
        gdf["capacity"] = np.nan
    gdf["capacity"] = pd.to_numeric(gdf["capacity"], errors="coerce")
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

    if len(gdf) > 0:
        gdf = impute_services(gdf, service)
        blocks_service, excess_services = aggregate_objects(
            blocks,
            gdf[["geometry", "capacity"]].copy(),
        )
    else:
        blocks_service = blocks[["geometry"]].copy()
        blocks_service["capacity"] = 0.0
        blocks_service["count"] = 0
        excess_services = gdf

    blocks_service = blocks_service.rename(columns={"count": f"count_{service}"})
    services[service] = gdf
    blocks_services[service] = blocks_service

    save_geo(gdf, SERVICE_DIR / f"{service}.geojson")
    save_geo(
        blocks_service,
        BLOCK_DIR / f"blocks_{service}.geojson",
        block_level=True,
    )

    capacity_total = float(gdf["capacity"].fillna(0).sum()) if len(gdf) > 0 else 0.0
    print(
        f"{service}: объектов={len(gdf):,}, вместимость={capacity_total:,.0f}, "
        f"вне кварталов={len(excess_services):,}"
    )


## 8. Конкурентная обеспеченность

Для каждого сервиса и каждого графа решается задача распределения спроса между кварталами:

- спрос считается по населению и нормативу на 1000 жителей;
- вместимость сервиса ограничена;
- учитывается конкуренция кварталов за близкие объекты;
- `provision_strong` — доля спроса, удовлетворённая в пределах норматива доступности;
- `provision_weak` — доля спроса с учётом связей за пределами норматива (при `max_depth=1` часто близка к strong).

**Единицы:** `provision_*` — доля от 0 до 1 (умножьте на 100, чтобы получить проценты). `demand_<service>` — спрос в чел., `capacity_<service>` — вместимость в местах/посещениях.

Нормативы берутся из `service_types_config` BlocksNet:

- школа: 120 мест на 1000 жителей, 15 минут;
- детский сад: 61 место на 1000 жителей, 7 минут;
- поликлиника: 13 посещений на 1000 жителей, 15 минут.

Для решения используется PuLP/CBC. Если для какого-то сервиса нет объектов, обеспеченность будет нулевой, но расчёт продолжится.


In [ ]:
provision_dfs = {}
provision_links = {}

for service in SERVICES:
    config = service_types_config[service]
    demand_norm = int(config["demand"])
    accessibility_norm = int(config["accessibility"])

    for graph_type in GRAPH_TYPES:
        data = blocks_buildings[["population"]].copy()
        data["capacity"] = (
            blocks_services[service]["capacity"]
            .reindex(data.index)
            .fillna(0)
        )
        data["population"] = data["population"].fillna(0)
        data["capacity"] = np.rint(data["capacity"]).astype(int)

        provision_df, links_df = competitive_provision(
            data,
            matrices_capped[graph_type],
            accessibility=accessibility_norm,
            demand=demand_norm,
            self_supply=True,
            max_depth=1,
        )

        rename_map = {
            "demand": f"demand_{service}",
            "capacity": f"capacity_{service}",
            "demand_left": f"demand_left_{service}",
            "demand_within": f"demand_within_{service}",
            "demand_without": f"demand_without_{service}",
            "capacity_left": f"capacity_left_{service}",
            "capacity_within": f"capacity_within_{service}",
            "capacity_without": f"capacity_without_{service}",
            "provision_strong": f"provision_strong_{service}",
            "provision_weak": f"provision_weak_{service}",
        }
        provision_df = provision_df.rename(columns=rename_map)
        provision_df = provision_df.drop(columns=["population"], errors="ignore")
        provision_dfs[(service, graph_type)] = provision_df
        provision_links[(service, graph_type)] = links_df

        strong = provision_df[f"provision_strong_{service}"].dropna()
        require(
            ((strong >= 0) & (strong <= 1 + 1e-9)).all(),
            f"{service}/{graph_type}: provision_strong вне диапазона [0, 1]",
        )

        links_path = TABLE_DIR / f"provision_links_{service}_{graph_type}.csv"
        links_df.reset_index().to_csv(links_path, index=False, encoding="utf-8-sig")

        save_geo(
            blocks[["geometry"]].join(provision_df),
            RESULT_DIR / f"provision_{service}_{graph_type}.geojson",
            block_level=True,
        )

        print(
            f"{service}/{graph_type}: "
            f"спрос={provision_df[f'demand_{service}'].sum():,.0f}, "
            f"вместимость={provision_df[f'capacity_{service}'].sum():,.0f}, "
            f"strong(среднее)={strong.mean():.3f}, "
            f"weak(среднее)={provision_df[f'provision_weak_{service}'].mean():.3f}"
        )


## 9. Плотность и морфотипы

Для каждого квартала считаются:

| Показатель | Формула | Единицы | Что показывает |
|---|---|---|---|
| **FSI** | площадь пола / площадь участка | м²/м² | Интенсивность застройки |
| **GSI** | площадь застройки / площадь участка | м²/м² | Долю участка под зданиями |
| **MXI** | жилая площадь / площадь пола | м²/м² | Долю жилой функции |
| **L** | площадь пола / площадь застройки | этажи | Среднюю этажность |
| **OSR** | (площадь участка − площадь застройки) / площадь пола | м²/м² | Обеспеченность открытыми пространствами |
| **share_living** | жилая площадь / площадь застройки | отношение | Условная доля жилой площади |
| **share_non_living** | нежилая площадь / площадь застройки | отношение | Условная доля нежилой площади |

Морфотипы:

- **Spacematrix** — кластеризация кварталов по FSI, MXI и этажности;
- **Strelka** — правило-ориентированная классификация по этажности, плотности и доле жилья.

Если в городе мало застроенных кварталов, число кластеров автоматически уменьшается.


In [ ]:
density_indicators = calculate_density_indicators(blocks_buildings)

density_result = blocks_buildings[["population", "count_buildings"]].join(
    density_indicators[
        ["fsi", "gsi", "mxi", "l", "osr", "share_living", "share_non_living"]
    ]
)

require(
    not (density_indicators[["fsi", "gsi", "mxi", "l"]] < 0).any().any(),
    "Индикаторы плотности содержат отрицательные значения",
)
if density_indicators["gsi"].max() > 1.0001:
    warn("GSI больше 1; возможно, площадь застройки превышает площадь квартала.")

developed_count = int((density_indicators["fsi"] > 0).sum())
if developed_count > 0:
    n_clusters = min(N_CLUSTERS, developed_count)
    spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(
        density_indicators,
        n_clusters=n_clusters,
        random_state=RANDOM_STATE,
    )
    spacematrix_result = spacematrix_df[["cluster", "morphotype"]].rename(
        columns={"morphotype": "morphotype_spacematrix"}
    )
    print(f"Застроенных кварталов: {developed_count:,}; кластеров Spacematrix: {n_clusters}")
else:
    warn("Нет кварталов с FSI > 0; морфотипы Spacematrix не определены.")
    spacematrix_result = pd.DataFrame(index=density_indicators.index)
    spacematrix_result["cluster"] = np.nan
    spacematrix_result["morphotype_spacematrix"] = None
    spacematrix_clusters = pd.DataFrame()

strelka_df = get_strelka_morphotypes(density_indicators)
strelka_result = strelka_df[["morphotype"]].rename(
    columns={"morphotype": "morphotype_strelka"}
)

blocks_base = blocks_with_land_use.join(density_result).join(spacematrix_result).join(strelka_result)
blocks_base["site_area"] = blocks_base.area

for service in SERVICES:
    blocks_base = blocks_base.join(blocks_services[service][[f"count_{service}"]])

require(blocks_base.index.equals(blocks.index), "Индексы кварталов и индикаторов не совпадают")

print(f"FSI: медиана={density_indicators['fsi'].median():.3f}, максимум={density_indicators['fsi'].max():.3f}")
print(f"GSI: медиана={density_indicators['gsi'].median():.3f}, максимум={density_indicators['gsi'].max():.3f}")
print(f"MXI: медиана={density_indicators['mxi'].median():.3f}")
print(f"L: медиана={density_indicators['l'].median():.2f} этажей")
print("Морфотипы Strelka:")
print(strelka_result["morphotype_strelka"].value_counts(dropna=False).head(15).to_string())

save_geo(
    blocks_base,
    RESULT_DIR / "blocks_indicators.geojson",
    block_level=True,
)

plot_map(
    blocks_base,
    column="fsi",
    title="FSI: коэффициент застройки",
    path=FIGURE_DIR / "fsi.png",
    cmap="magma_r",
)
plot_map(
    blocks_base,
    column="gsi",
    title="GSI: коэффициент покрытия",
    path=FIGURE_DIR / "gsi.png",
    cmap="viridis",
)
if blocks_base["morphotype_strelka"].notna().any():
    plot_map(
        blocks_base,
        column="morphotype_strelka",
        title="Морфотипы Strelka",
        path=FIGURE_DIR / "morphotypes_strelka.png",
        legend=False,
    )


## 10. Зональная доступность

Зональная доступность (`area_accessibility`) — средневзвешенное время поездки из квартала до всех кварталов города, где вес — площадь квартала назначения. **Единица — минуты.** Чем меньше значение, тем лучше транспортная/пешеходная связность.

Считается отдельно для `drive` и `walk`. Недостижимые пары уже заменены на `MAX_TRAVEL_MIN = 180` минут.


In [ ]:
area_accs = {}

for graph_type in GRAPH_TYPES:
    area_acc = area_accessibility(
        matrices_capped[graph_type],
        blocks_buildings[["site_area"]],
    )
    area_acc = area_acc.rename(
        columns={"area_accessibility": f"area_accessibility_{graph_type}"}
    )
    area_accs[graph_type] = area_acc

    save_geo(
        blocks[["geometry"]].join(area_acc),
        RESULT_DIR / f"area_accessibility_{graph_type}.geojson",
        block_level=True,
    )

    values = area_acc[f"area_accessibility_{graph_type}"]
    print(
        f"{graph_type}: area_accessibility "
        f"медиана={values.median():.2f} мин, "
        f"95-й процентиль={values.quantile(0.95):.2f} мин, "
        f"максимум={values.max():.2f} мин"
    )


## 11. Итоговые результаты

Для каждого режима передвижения собирается единый слой кварталов, в котором есть:

- характеристики застройки и населения;
- `land_use`;
- морфотипы;
- площадь и плотностные индикаторы;
- зональная доступность;
- обеспеченность школами, детскими садами и поликлиниками.

Итоговые слои `blocks_indicators_drive.geojson` и `blocks_indicators_walk.geojson` — основа для будущего HTML-дашборда.


In [ ]:
graph_indicators = {}

for graph_type in GRAPH_TYPES:
    result = blocks_base.join(area_accs[graph_type])
    for service in SERVICES:
        result = result.join(provision_dfs[(service, graph_type)])

    graph_indicators[graph_type] = result
    save_geo(
        result,
        RESULT_DIR / f"blocks_indicators_{graph_type}.geojson",
        block_level=True,
    )
    print(
        f"{graph_type}: сохранён слой с {len(result.columns)} атрибутами "
        f"для {len(result):,} кварталов"
    )


### 11.1. Сводная таблица и карты

`summary.csv` содержит описательные статистики по показателям: количество кварталов, сумму, среднее, медиану, минимум и максимум. Для каждого графа и сервиса строки помечены в колонках `graph` и `service`.

PNG-карты сохраняются в `examples/outputs/nizhnevartovsk/figures/`.


In [ ]:
summary_rows = []


def add_summary(metric, series, graph="common", service=""):
    values = (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    summary_rows.append(
        {
            "metric": metric,
            "graph": graph,
            "service": service,
            "count": int(values.size),
            "sum": float(values.sum()) if values.size else np.nan,
            "mean": float(values.mean()) if values.size else np.nan,
            "median": float(values.median()) if values.size else np.nan,
            "min": float(values.min()) if values.size else np.nan,
            "max": float(values.max()) if values.size else np.nan,
        }
    )


for metric in ["population", "count_buildings", "fsi", "gsi", "mxi", "l", "osr", "share_living", "share_non_living"]:
    add_summary(metric, blocks_base[metric])

for service in SERVICES:
    add_summary(f"count_{service}", blocks_base[f"count_{service}"])

for graph_type in GRAPH_TYPES:
    result = graph_indicators[graph_type]
    add_summary(
        f"area_accessibility_{graph_type}",
        result[f"area_accessibility_{graph_type}"],
        graph=graph_type,
    )
    for service in SERVICES:
        add_summary(
            f"provision_strong_{service}",
            result[f"provision_strong_{service}"],
            graph=graph_type,
            service=service,
        )
        add_summary(
            f"provision_weak_{service}",
            result[f"provision_weak_{service}"],
            graph=graph_type,
            service=service,
        )

summary_df = pd.DataFrame(summary_rows)
summary_path = TABLE_DIR / "summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"Сводная таблица: {summary_path.relative_to(PROJECT_DIR)}")
print(summary_df.head(20).to_string())


if blocks_base["population"].fillna(0).sum() > 0:
    plot_map(
        blocks_base,
        column="population",
        title="Население по кварталам, чел.",
        path=FIGURE_DIR / "population_blocks.png",
        cmap="YlOrRd",
    )

if blocks_base["fsi"].max() > 0:
    plot_map(
        blocks_base,
        column="fsi",
        title="FSI",
        path=FIGURE_DIR / "fsi_final.png",
        cmap="magma_r",
    )

if blocks_base["morphotype_strelka"].notna().any():
    plot_map(
        blocks_base,
        column="morphotype_strelka",
        title="Морфотипы Strelka",
        path=FIGURE_DIR / "morphotypes_final.png",
        legend=False,
    )

for graph_type in GRAPH_TYPES:
    result = graph_indicators[graph_type]
    plot_map(
        result,
        column=f"area_accessibility_{graph_type}",
        title=f"Зональная доступность ({graph_type}), мин",
        path=FIGURE_DIR / f"area_accessibility_{graph_type}.png",
        cmap="magma_r",
    )
    for service in SERVICES:
        if result[f"provision_strong_{service}"].notna().any():
            plot_map(
                result,
                column=f"provision_strong_{service}",
                title=f"Обеспеченность: {service} ({graph_type})",
                path=FIGURE_DIR / f"provision_{service}_{graph_type}.png",
                cmap="RdYlGn",
                vmin=0,
                vmax=1,
            )


## 12. Манифест и финальные проверки

Манифест фиксирует параметры запуска, версии, число кварталов, население, размеры графов и наличие ключевых файлов.

Финальная проверка убеждается, что:

- все ожидаемые GeoJSON, Parquet, CSV и Pickle созданы;
- матрицы можно прочитать повторно и их размер корректен;
- сводная таблица и манифест записаны.

Если какая-то проверка не прошла, ноутбук выведет список отсутствующих файлов.


In [ ]:
manifest = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "territory_query": TERRITORY_QUERY,
    "area_km2": float(area_km2),
    "crs": crs.to_string(),
    "graph_source": "OpenStreetMap (Overpass API) via IduEdu",
    "travel_time_unit": "minutes",
    "blocks": int(len(blocks)),
    "population_estimate": float(blocks_buildings["population"].sum()),
    "parameters": {
        "graph_types": list(GRAPH_TYPES),
        "services": list(SERVICES),
        "max_travel_min": MAX_TRAVEL_MIN,
        "n_clusters": N_CLUSTERS,
        "random_state": RANDOM_STATE,
        "default_living_demand": DEFAULT_LIVING_DEMAND,
        "use_cache": USE_CACHE,
        "redownload": REDOWNLOAD,
    },
    "versions": {
        "blocksnet": getattr(blocksnet, "__version__", "unknown"),
        "iduedu": getattr(iduedu, "__version__", "unknown"),
        "osmnx": ox.__version__,
        "geopandas": gpd.__version__,
        "pandas": pd.__version__,
        "networkx": nx.__version__,
    },
    "graphs": {
        graph_type: {
            "mode": graph_type,
            "nodes": int(graphs[graph_type].number_of_nodes()),
            "edges": int(graphs[graph_type].number_of_edges()),
        }
        for graph_type in GRAPH_TYPES
    },
    "services": {
        service: {
            "objects": int(len(services[service])),
            "capacity": float(services[service]["capacity"].fillna(0).sum()),
        }
        for service in SERVICES
    },
}

manifest_path = DATA_DIR / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Манифест: {manifest_path.relative_to(PROJECT_DIR)}")

expected_files = [
    RAW_DIR / "boundaries.geojson",
    RAW_DIR / "water.geojson",
    RAW_DIR / "roads.geojson",
    RAW_DIR / "railways.geojson",
    RAW_DIR / "functional_zones.geojson",
    RAW_DIR / "buildings.geojson",
    RAW_DIR / "buildings_imputed.geojson",
    BLOCK_DIR / "blocks.geojson",
    BLOCK_DIR / "blocks_land_use.geojson",
    BLOCK_DIR / "blocks_buildings.geojson",
    RESULT_DIR / "blocks_indicators.geojson",
    summary_path,
    manifest_path,
]

for service in SERVICES:
    expected_files.extend(
        [
            RAW_DIR / f"service_{service}.geojson",
            SERVICE_DIR / f"{service}.geojson",
            BLOCK_DIR / f"blocks_{service}.geojson",
        ]
    )

for graph_type in GRAPH_TYPES:
    expected_files.extend(
        [
            GRAPH_DIR / f"graph_{graph_type}.pickle",
            MATRIX_DIR / f"accessibility_matrix_{graph_type}.parquet",
            RESULT_DIR / f"area_accessibility_{graph_type}.geojson",
            RESULT_DIR / f"blocks_indicators_{graph_type}.geojson",
        ]
    )
    for service in SERVICES:
        expected_files.append(
            RESULT_DIR / f"provision_{service}_{graph_type}.geojson"
        )
        expected_files.append(
            TABLE_DIR / f"provision_links_{service}_{graph_type}.csv"
        )

missing = [
    str(path.relative_to(PROJECT_DIR))
    for path in expected_files
    if not path.exists()
]
require(not missing, f"Не созданы ожидаемые файлы: {missing}")

for graph_type in GRAPH_TYPES:
    matrix_path = MATRIX_DIR / f"accessibility_matrix_{graph_type}.parquet"
    matrix = pd.read_parquet(matrix_path)
    require(
        matrix.shape == (len(blocks), len(blocks)),
        f"{graph_type}: повторное чтение матрицы дало размер {matrix.shape}",
    )

print(f"Проверено файлов: {len(expected_files)}")
print(f"Кварталов: {len(blocks):,}")
print(f"Население: {blocks_buildings['population'].sum():,.0f} чел.")
print("Все финальные проверки пройдены.")


## 13. Как перезапускать и что делать при проблемах

### Обычный повторный запуск

Оставьте `USE_CACHE = True`. Ноутбук возьмёт уже скачанные OSM-слои, графы и матрицы из `examples/data/nizhnevartovsk/`. Это самый быстрый режим.

### Принудительное обновление данных

Установите `REDOWNLOAD = True` и выполните ноутбук сверху вниз. Будут заново запрошены OSM-слои, графы и матрицы. Либо удалите папку:

```powershell
Remove-Item -Recurse -Force examples\data\nizhnevartovsk
```

### Если геокодер вернул не тот район

1. Проверьте `area_km2` и карту `boundaries.png`.
2. Уточните `TERRITORY_QUERY`, например указав район или добавив `Россия`.
3. Удалите `examples/data/nizhnevartovsk/raw/boundaries.geojson` и запустите ячейку границ заново.

### Если Overpass долго не отвечает

Повторите запуск ячейки: Overpass иногда перегружен. Если ошибка повторяется, уменьшите территорию или используйте выгрузку OSM и локальный `.osm.pbf` (потребуется изменить способ загрузки).

### Если матрица почти полностью `inf`

Возможные причины:

- граница включает удалённые районы без дорожной сети;
- слишком маленький `MAX_TRAVEL_MIN`;
- граф `walk` фрагментирован водоёмами и железными дорогами.

Ноутбук продолжит работу с штрафом `MAX_TRAVEL_MIN`, но интерпретировать доступность нужно осторожно. Увеличьте `MAX_TRAVEL_MIN` или уточните границу.

### Если нет Jupyter-ядра

```powershell
.\.venv\Scripts\Activate.ps1
python -m pip install -e ".[dev]"
python -m ipykernel install --user --name python-package-boilerplate --display-name "Python (.venv)"
```

### Если не устанавливается BlocksNet или не хватает `torch`

```powershell
.\.venv\Scripts\Activate.ps1
python -m pip install -e ".[dev]"
```

В `pyproject.toml` уже указан `blocksnet[ipynb,torch]` из git-тега `v1.0.0.a9`. Если ошибка сохраняется, обновите `pip` и повторите установку:

```powershell
python -m pip install --upgrade pip
python -m pip install -e ".[dev]"
```

### Где смотреть результаты

- `examples/outputs/nizhnevartovsk/tables/summary.csv` — сводные статистики;
- `examples/outputs/nizhnevartovsk/figures/` — карты;
- `examples/data/nizhnevartovsk/results/blocks_indicators_drive.geojson` — автомобильный слой;
- `examples/data/nizhnevartovsk/results/blocks_indicators_walk.geojson` — пешеходный слой;
- `examples/data/nizhnevartovsk/manifest.json` — параметры и версии.
